In [3]:
import os  # noqa
import shutil  # noqa
import urllib  # noqa
from pathlib import Path  # noqa

# url = "https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz"
# urllib.request.urlretrieve(url, file_path)

external_data_path = Path(
    "/home/ds13/.bookmarks/shared-project-assets/Programming__/GitHub_datasets"
)
filename = "ghtorrent-2019-01-07_dataset"

data_path = Path("data")

data_path.mkdir(exist_ok=True)

raw_data_path = data_path / "raw"
raw_data_path.mkdir(exist_ok=True)

dataset_path = raw_data_path / filename

if not dataset_path.exists():
    shutil.unpack_archive(
        external_data_path / (filename + ".zip"), extract_dir=raw_data_path
    )
    # file_path.unlink()  # Remove archive after extracting it.

In [4]:
import pandas as pd  # noqa

CHUNKSIZE = 100_000

dataset_file_path = dataset_path / "ghtorrent-2019-01-07.csv"
data = pd.read_csv(dataset_file_path, chunksize=CHUNKSIZE, on_bad_lines='warn', sep=',')
data_part1 = pd.read_csv(dataset_file_path, nrows=CHUNKSIZE)

In [5]:
from IPython.display import display  # noqa

# df = data
# with df as reader:
    # data_chunk = reader.get_chunk(CHUNKSIZE)
    # display(data_chunk.head())

In [6]:
data_part1.head()

,actor_login,actor_id,comment_id,comment,repo,language,author_login,author_id,pr_id,c_id,commit_date
0,nikolagjorgjievski,34368385,251381442,Do you think its better if we use cards here i...,journal-app,NaN,TamaraStankovska,13293313,54298419,1282815601,2019-01-07 17:11:19 UTC
1,nikolagjorgjievski,34368385,246438304,Fix indent of \transition name=\modal\\,journal-app,NaN,TamaraStankovska,13293313,53137423,1263914484,2019-01-07 19:32:35 UTC
2,tn3rb,2327533,248888072,y u no ternary ?\\the previous 12 lines could ...,event-espresso-core,PHP,joshfeck,1377750,49682118,1274046759,2019-01-07 22:29:38 UTC
3,caalador,991111,198711352,`getRel`,flow,Java,pleku,1621377,41569148,1054030538,2019-01-07 05:53:13 UTC
4,XuHuaiyu,5620059,277629250,"Gotcha, never mind.",tidb,Go,XuHuaiyu,5620059,60396802,1371118363,2019-01-07 11:19:59 UTC


In [7]:
data_part1.shape

(100000, 11)

In [8]:
data_part1.dtypes

actor_login     object
actor_id         int64
comment_id       int64
comment         object
repo            object
language        object
author_login    object
author_id        int64
pr_id            int64
c_id             int64
commit_date     object
dtype: object

Некоторые колонки имеют неточные типы данных, их следует преобразовать

In [9]:
data_part1["commit_date"] = pd.to_datetime(data_part1["commit_date"], utc=True)
data_part1 = data_part1.astype(
    {
        "actor_id": "uint64",
        "comment_id": "uint64",
        "author_id": "uint64",
        "pr_id": "uint64",
        "c_id": "uint64",
    }
)
data_part1.dtypes, data_part1["commit_date"]

(actor_login                  object
 actor_id                     uint64
 comment_id                   uint64
 comment                      object
 repo                         object
 language                     object
 author_login                 object
 author_id                    uint64
 pr_id                        uint64
 c_id                         uint64
 commit_date     datetime64[ns, UTC]
 dtype: object,
 0       2019-01-07 17:11:19+00:00
 1       2019-01-07 19:32:35+00:00
 2       2019-01-07 22:29:38+00:00
 3       2019-01-07 05:53:13+00:00
 4       2019-01-07 11:19:59+00:00
                    ...           
 99995   2019-01-07 08:53:18+00:00
 99996   2019-01-07 23:37:32+00:00
 99997   2019-01-07 11:08:34+00:00
 99998   2019-01-07 09:43:41+00:00
 99999   2019-01-07 11:45:55+00:00
 Name: commit_date, Length: 100000, dtype: datetime64[ns, UTC])

Посмотрим, как можно оптимизировать наш большой набор данных.

In [10]:
data_part1.memory_usage(deep=True)

Index                128
actor_login      6582791
actor_id          800000
comment_id        800000
comment         16200000
repo             6595877
language         5530996
author_login     6606362
author_id         800000
pr_id             800000
c_id              800000
commit_date       800000
dtype: int64

Как видно, наибольший размер занимают строки. Заметим, что исходя из знаний о предметной области,
многие из них будут часто повторяться. Комментарии, как правило отличны, в то время как
пользователи и репозитории в которых они оставляют комментарии, будут повторяться.

Языки и вовсе являются ограниченным множеством.

Трансформируем соответствующие колонки в категориальный тип.

In [11]:
data_part1 = data_part1.astype(
    {
        "actor_login": "category",
        "language": "category",
        "repo": "category",
        "author_login": "category",
    }
)
data_part1.dtypes, data_part1.memory_usage(deep=True)

(actor_login                category
 actor_id                     uint64
 comment_id                   uint64
 comment                      object
 repo                       category
 language                   category
 author_login               category
 author_id                    uint64
 pr_id                        uint64
 c_id                         uint64
 commit_date     datetime64[ns, UTC]
 dtype: object,
 Index                128
 actor_login      2046392
 actor_id          800000
 comment_id        800000
 comment         16200000
 repo              679864
 language          107737
 author_login     1049744
 author_id         800000
 pr_id             800000
 c_id              800000
 commit_date       800000
 dtype: int64)

Как видно, наша гипотеза оказалась верной, все колонки кроме 'actor_login'
стали занимать меньше места в памяти. Пока что 'actor_login' оставим
категориальным, так как разница в памяти относительно невелика.

In [12]:
# Проверим наличие пустых значений
# Цикл по колонкам датасета
columns_number_of_empty_columns: dict[str, int] = {}

chunks1 = data
with data as reader:
    data_chunk = reader.get_chunk(CHUNKSIZE)

    for col in data_chunk.columns:
        # Количество пустых значений - все значения заполнены
        temp_null_count = data_chunk[data_chunk[col].isnull()].shape[0]
        columns_number_of_empty_columns[col] = (
            columns_number_of_empty_columns.get(col, 0) + temp_null_count
        )


pd.DataFrame(
    data={
        "Колонка": columns_number_of_empty_columns.keys(),
        "Кол-во пустых значений": columns_number_of_empty_columns.values(),
    }
)

,Колонка,Кол-во пустых значений
0,actor_login,0
1,actor_id,0
2,comment_id,0
3,comment,1
4,repo,0
5,language,23051
6,author_login,15
7,author_id,0
8,pr_id,0
9,c_id,0


> Далее набор данных `data` будет выкидывать ошибку. Это связано с тем, что
итератор закончил свою работу. Нужно заново читать набор данных.